In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/lmsys-chatbot-arena/sample_submission.csv
/kaggle/input/competitions/lmsys-chatbot-arena/train.csv
/kaggle/input/competitions/lmsys-chatbot-arena/test.csv


This competition challenges you to predict which responses users will prefer in a head-to-head battle between chatbots powered by large language models (LLMs). You'll be given a dataset of conversations from the Chatbot Arena, where different LLMs generate answers to user prompts. By developing a winning machine learning model, you'll help improve how chatbots interact with humans and ensure they better align with human preferences.

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import paired_cosine_distances
from scipy.sparse import hstack, csr_matrix
from lightgbm import LGBMClassifier

In [3]:
nltk.download('nltk_data', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

[nltk_data] Error loading nltk_data: Package 'nltk_data' not found in
[nltk_data]     index


True

In [4]:
path = '/kaggle/input/competitions/lmsys-chatbot-arena/train.csv'
path_test = '/kaggle/input/competitions/lmsys-chatbot-arena/test.csv'

In [5]:
df_train = pd.read_csv(path, index_col='id')
df_test = pd.read_csv(path_test, index_col='id')

In [6]:
df_train.columns

Index(['model_a', 'model_b', 'prompt', 'response_a', 'response_b',
       'winner_model_a', 'winner_model_b', 'winner_tie'],
      dtype='object')

In [7]:
df_test.columns

Index(['prompt', 'response_a', 'response_b'], dtype='object')

In [8]:
df_train.head(5)

,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
id,,,,,,,,
30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [9]:
def encode_target(row):
    if row["winner_model_a"] == 1:
        return 0
    elif row["winner_model_b"] == 1:
        return 1
    else:
        return 2

df_train["target"] = df_train.apply(encode_target, axis=1)

## 2. Text Preprocessing

In [10]:
custom_stopwords = set(stopwords.words('english')) - {'not', 'no', 'nor', 'neither', 'never', 'none'}

In [11]:
def clean_and_tokenize(text: str, apply_stemming: bool = False) -> str:
    if not isinstance(text, str):
        return ""
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = word_tokenize(text.lower())
    
    filtered_tokens = [
        tok for tok in tokens 
        if tok.isalnum() and tok not in custom_stopwords
    ]
    if apply_stemming:
        stemmer = PorterStemmer()
        filtered_tokens = [stemmer.stem(tok) for tok in filtered_tokens]
        
    return " ".join(filtered_tokens)

In [12]:
df_train["prompt_clean"] = df_train["prompt"].apply(clean_and_tokenize)

In [13]:
df_test["prompt_clean"] = df_test["prompt"].apply(clean_and_tokenize)

In [14]:
df_train["response_a_clean"] = df_train["response_a"].apply(clean_and_tokenize)

In [15]:
df_test["response_a_clean"] = df_test["response_a"].apply(clean_and_tokenize)

In [16]:
df_train["response_b_clean"] = df_train["response_b"].apply(clean_and_tokenize)

In [17]:
df_test["response_b_clean"] = df_test["response_b"].apply(clean_and_tokenize)

In [18]:
df_train.head(5)

,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,target,prompt_clean,response_a_clean,response_b_clean
id,,,,,,,,,,,,
30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0,0,morally right try certain percentage females m...,question whether morally right aim certain per...,ai personal beliefs opinions however tell ques...
53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0,1,difference marriage license marriage certifica...,marriage license legal document allows couple ...,marriage license marriage certificate two diff...
65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1,2,explain function calling would call function,function calling process invoking executing fu...,function calling process invoking function pro...
96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0,0,create test set rare category want build class...,creating test set rare category challenging ma...,building classifier rare category creating tes...
198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0,1,best way travel jerusalem car bus plane,best way travel tel aviv jerusalem depends per...,best way travel jerusalem depends personal pre...


In [19]:
df_test.head(5)

,prompt,response_a,response_b,prompt_clean,response_a_clean,response_b_clean
id,,,,,,
136060,"[""I have three oranges today, I ate an orange ...","[""You have two oranges today.""]","[""You still have three oranges. Eating an oran...",three oranges today ate orange yesterday many ...,two oranges today,still three oranges eating orange yesterday no...
211333,"[""You are a mediator in a heated political deb...","[""Thank you for sharing the details of the sit...","[""Mr Reddy and Ms Blue both have valid points ...",mediator heated political debate two opposing ...,thank sharing details situation mediator under...,mr reddy ms blue valid points arguments one ha...
1233961,"[""How to initialize the classification head wh...","[""When you want to initialize the classificati...","[""To initialize the classification head when p...",initialize classification head transfer learni...,want initialize classification head transfer l...,initialize classification head performing tran...


#### 3. Additionnal features extracted for traditional ML

In [20]:
def extract_tabular_features(data):
    feats = pd.DataFrame()

    feats["char_len_a"] = data["response_a"].str.len()
    feats["char_len_b"] = data["response_b"].str.len()
    feats["char_diff"] = feats["char_len_a"] - feats["char_len_b"]
    feats["char_ratio"] = (feats["char_len_a"] + 1) / (feats["char_len_b"] + 1)

    feats["word_len_a"] = data["response_a"].apply(lambda x: len(x.split()))
    feats["word_len_b"] = data["response_b"].apply(lambda x: len(x.split()))
    feats["word_diff"] = feats["word_len_a"] - feats["word_len_b"]

    feats["has_code_a"] = data["response_a"].str.contains(r"```").astype(int)
    feats["has_code_b"] = data["response_b"].str.contains(r"```").astype(int)
    feats["has_list_a"] = data["response_a"].str.contains(r"^\s*[-*]\s+", regex=True).astype(int)
    feats["has_list_b"] = data["response_b"].str.contains(r"^\s*[-*]\s+", regex=True).astype(int)

    return feats

In [21]:
num_feats_df = extract_tabular_features(df_train)

In [22]:
num_feats_test = extract_tabular_features(df_test)

In [23]:
num_feats_df.head(3)

,char_len_a,char_len_b,char_diff,char_ratio,word_len_a,word_len_b,word_diff,has_code_a,has_code_b,has_list_a,has_list_b
id,,,,,,,,,,,
30192,4538,1206,3332,3.760563,656,204,452,0,0,0,0
53567,3114,3649,-535,0.853425,531,571,-40,0,0,0,0
65089,921,1835,-914,0.502179,138,280,-142,1,1,0,0


In [24]:
num_feats_test.head(3)

,char_len_a,char_len_b,char_diff,char_ratio,word_len_a,word_len_b,word_diff,has_code_a,has_code_b,has_list_a,has_list_b
id,,,,,,,,,,,
136060,31,114,-83,0.278261,5,19,-14,0,0,0,0
211333,1457,460,997,3.162690,217,75,142,0,0,0,0
1233961,3984,3716,268,1.072101,623,437,186,0,1,0,0


#### 4. Vectorization

In [25]:
tfidf = TfidfVectorizer(max_features=2500, stop_words="english", ngram_range=(1, 2))

In [26]:
combined_text = pd.concat([df_train["prompt"], df_train["response_a"], df_train["response_b"]])

In [27]:
tfidf.fit(combined_text)

TfidfVectorizer(max_features=2500, ngram_range=(1, 2), stop_words='english')

#### 7. Train and test data vectorization

In [28]:
p_vec = tfidf.transform(df_train["prompt"])

In [29]:
a_vec = tfidf.transform(df_train["response_a"])

In [30]:
b_vec = tfidf.transform(df_train["response_b"])

In [31]:
p_test_vec = tfidf.transform(df_test["prompt"])

In [32]:
a_test_vec = tfidf.transform(df_test["response_a"])

In [33]:
b_test_vec = tfidf.transform(df_test["response_b"])

#### 5. Similarity vectors for train and test

In [34]:
num_feats_df["sim_prompt_a"] = 1 - paired_cosine_distances(p_vec, a_vec)

In [35]:
num_feats_df["sim_prompt_b"] = 1 - paired_cosine_distances(p_vec, b_vec)

In [36]:
num_feats_df["sim_diff"] = num_feats_df["sim_prompt_a"] - num_feats_df["sim_prompt_b"]

In [37]:
num_sparse = csr_matrix(num_feats_df.values)

In [38]:
num_feats_test["sim_prompt_a"] = 1 - paired_cosine_distances(p_test_vec, a_test_vec)

In [39]:
num_feats_test["sim_prompt_b"] = 1 - paired_cosine_distances(p_test_vec, b_test_vec)

In [40]:
num_feats_test["sim_diff"] = num_feats_test["sim_prompt_a"] - num_feats_test["sim_prompt_b"]

In [41]:
num_sparse_test = csr_matrix(num_feats_test.values)

#### 6. Model and evaluation

In [42]:
X = hstack([a_vec, b_vec, num_sparse])

In [43]:
X_test = hstack([a_test_vec, b_test_vec, num_sparse_test])

In [44]:
type(X)

scipy.sparse._csr.csr_matrix

In [45]:
y = df_train["target"].values

In [46]:
clf = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    objective="multiclass",
    num_class=3,
    random_state=42
)

In [47]:
clf.fit(X, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.168442 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 978617
[LightGBM] [Info] Number of data points in the train set: 57477, number of used features: 4997
[LightGBM] [Info] Start training from score -1.052458
[LightGBM] [Info] Start training from score -1.073206
[LightGBM] [Info] Start training from score -1.174380


LGBMClassifier(learning_rate=0.05, n_estimators=300, num_class=3,
               objective='multiclass', random_state=42)

In [48]:
test_proba = clf.predict_proba(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [49]:
sample_submission = pd.read_csv(
    "/kaggle/input/competitions/lmsys-chatbot-arena/sample_submission.csv"
)

submission = sample_submission.copy()

submission["winner_model_a"] = test_proba[:, 0]
submission["winner_model_b"] = test_proba[:, 1]
submission["winner_tie"] = test_proba[:, 2]

In [50]:
submission = sample_submission.copy()

# 3. Assign predicted probabilities to target columns
# Assumes test_proba has shape (N, 3) where:
# Column 0 = winner_model_a
# Column 1 = winner_model_b
# Column 2 = winner_tie
submission["winner_model_a"] = test_proba[:, 0]
submission["winner_model_b"] = test_proba[:, 1]
submission["winner_tie"] = test_proba[:, 2]

# 4. Save to submission.csv in the root working directory
submission.to_csv("submission.csv", index=False)

print("Submission file successfully generated!")
print(submission.head())

Submission file successfully generated!
        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.192959        0.458923    0.348117
1   211333        0.498890        0.261683    0.239426
2  1233961        0.283045        0.409424    0.307531
